# 03 - Deduplicaton

In [1]:
ENV = "local"

from pathlib import Path

if ENV == 'colab':
    DATA_DIR = Path('/content/drive/MyDrive/thesis')
else:
    DATA_DIR = Path('00_data')

if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')


### Imports

In [7]:
import json
import re
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import orjson
import pandas as pd
from pandas import json_normalize
import matplotlib.pyplot as plt


import pyarrow as pa
import pyarrow.parquet as pq

from rapidfuzz import fuzz, process
from sentence_transformers import SentenceTransformer


from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, average_precision_score,
    classification_report, roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_val_score,
    cross_validate, train_test_split,
)

from xgboost import XGBClassifier

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [3]:
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from general.util.banned_words import add_flagged_column_vectorized

## data.gov

In [4]:
dgf = pd.read_parquet(str(DATA_DIR / '02_dgf.parquet'))
print(f"dgf shape: {dgf.shape}")
dgf

dgf shape: (514352, 28)


,title,description,publisher,identifier,slug,has_spatial,popularity,last_harvested_date,keyword,theme,...,dcat_access_comment,dcat_landing_page,dcat_language,dcat_temporal,dcat_spatial,dcat_bureau_code,dcat_program_code,dcat_periodicity,dcat_license,dcat_rights
0,Lottery Powerball Winning Numbers: Beginning 2010,Go to http://on.ny.gov/1GpWiHD on the New York...,data.ny.gov,https://data.ny.gov/api/views/d6yy-54nr,lottery-powerball-winning-numbers-beginning-2010,False,8525,2026-02-21T20:06:54.833664,"[new york lottery, powerball, results, winning]",[Government & Finance],...,,https://data.ny.gov/d/d6yy-54nr,[],,,[],[],,,
1,Electric Vehicle Population Data,This dataset shows the Battery Electric Vehicl...,data.wa.gov,https://data.wa.gov/api/views/f6w7-q2d2,electric-vehicle-population-data,False,6797,2026-02-14T15:35:27.252171,"[bev, bevs, bolt, car, cars, chevrolet, chevy,...",[Transportation],...,,https://data.wa.gov/d/f6w7-q2d2,[],,,[],[],,http://opendatacommons.org/licenses/odbl/1.0/,
2,FIMA NFIP Redacted Claims (OpenFEMA),Congress passed the National Flood Insurance A...,FEMA/Response and Recovery/Recovery Directorate,FEMA-0397,fima-nfip-redacted-claims-openfema,False,5354,2025-09-08T02:15:46.025831,[Assets],[],...,,,[],,,[024:070],[024:000],,https://www.usa.gov/government-works,
3,Supply Chain Greenhouse Gas Emission Factors v...,The datasets comprise greenhouse gas (GHG) emi...,U.S. EPA Office of Research and Development (ORD),https://doi.org/10.23719/1531143,supply-chain-greenhouse-gas-emission-factors-v...,False,4846,2025-08-02T20:59:33.260223,"[GHG reporting, USEEIO, climate change, indust...",[],...,,,[],,,[020:00],[020:000],,https://pasteur.epa.gov/license/sciencehub-lic...,
4,Crime Data from 2020 to Present,******Notice: Transition to NIBRS-Compliant Cr...,data.lacity.org,https://data.lacity.org/api/views/2nrs-mtv8,crime-data-from-2020-to-present,False,4530,2026-01-03T19:36:19.148715,"[crime, crime data, crimes, lapd, police, safe...",[Public Safety],...,,https://data.lacity.org/d/2nrs-mtv8,[],,,[],[],,http://creativecommons.org/publicdomain/zero/1...,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514347,Temperature profile data collected using bathy...,Temperature profile data were collected using ...,NOAA National Centers for Environmental Inform...,gov.noaa.nodc:8200029,temperature-profile-data-collected-using-bathy...,True,0,2026-01-26T23:27:42.564041,"[8200029, water depth, WATER TEMPERATURE, bath...",[],...,,https://www.ncei.noaa.gov/contact,[],1977-03-07T00:00:00+00:00/1979-03-31T00:00:00+...,"175.55,37.0,165.9,50.5",[],[],,https://creativecommons.org/publicdomain/zero/...,otherRestrictions
514348,"TIGER/Line Shapefile, Current, County, Howard ...",The TIGER/Line shapefiles and related database...,"U.S. Department of Commerce, U.S. Census Burea...",tl_2025_29089_addrfeat.shp.iso.xml,tiger-line-shapefile-current-county-howard-cou...,True,0,2026-01-27T08:36:10.449813,"[County or equivalent entity, Table, State FIP...",[],...,,,[],2025-06-01T00:00:00+00:00/2026-10-01T00:00:00+...,"-92.430229,38.967537,-92.945317,39.343586",[],[],,https://creativecommons.org/publicdomain/zero/...,otherRestrictions
514349,"TIGER/Line Shapefile, 2019, county, Hall Coun...",The TIGER/Line shapefiles and related database...,"U.S. Department of Commerce, U.S. Census Burea...",tl_2019_31079_addr.dbf.shp.iso.xml,tiger-line-shapefile-2019-county-hall-county-n...,True,0,2026-01-28T23:32:59.537901,"[State FIPS Code, County FIPS Code, State GNIS...",[],...,,,[],2018-06-01T00:00:00+00:00/2019-05-01T00:00:00+...,"-98.282353,40.698284,-98.721975,41.04674",[],[],,https://creativecommons.org/publicdomain/zero/...,otherRestrictions
514350,EPA FRS Facilities Single File CSV Download f...,The Facility Registry System (FRS) identifies ...,U.S. EPA Office of Environmental Information (...,68C41345-7798-4193-85EF-8DD0FEC25A72,epa-frs-facilities-single-file-csv-download-fo...,True,0,2025-08-08T20:46:44.482572,"[A

## data.gov mirror

In [5]:
dgm = pd.read_parquet('00_data/02_dgm.parquet')
dgm

,id,url,description,dgm_id,dgm_name,dgm_title,dgm_notes,dgm_state,dgm_type,dgm_url,...,org_id,org_name,org_title,org_type,org_description,org_created,org_is_organization,org_approval_status,org_state,tags
0,1ce2d0a1-8c2d-47a2-89cb-b68122ff0099,https://catalog.data.gov/dataset/afsc-race-sap...,"Archive of data.gov dataset ""AFSC/RACE/SAP/Arm...",36a5bb7a-c948-40ee-9272-61e7833f7041,afsc-race-sap-armistead-1975-2016-eastern-beri...,AFSC/RACE/SAP/Armistead: 1975 - 2016 eastern B...,The Resource Assessment and Conservation Engin...,active,dataset,None,...,5f4f1195-e770-4a2a-8f75-195cd98860ce,noaa-gov,National Oceanic and Atmospheric Administratio...,organization,,2020-11-10T15:36:13.098184,True,approved,active,"alaska,alaska fisheries science center,bering ..."
1,0d24350b-ae44-49d1-9ebd-8fd5d5b3ed4f,https://catalog.data.gov/dataset/tiger-line-sh...,"Archive of data.gov dataset ""TIGER/Line Shapef...",4106992e-d05e-49d1-ab5f-21c9d109d8b2,tiger-line-shapefile-2023-county-moca-municipi...,"TIGER/Line Shapefile, 2023, County, Moca Munic...",The TIGER/Line shapefiles and related database...,active,dataset,None,...,fb3131aa-ef06-4a00-ad84-67d93a71d7e3,census-gov,"U.S. Census Bureau, Department of Commerce",organization,The Census Bureau's mission is to serve as the...,2020-11-10T14:08:17.917195,True,approved,active,"72099,area hydrography identifier,county fips ..."
2,c464f1c5-b1e7-4252-8674-686ce9d46c84,https://catalog.data.gov/dataset/fbsab-recruit...,"Archive of data.gov dataset ""FBSAB RECRUIT Ree...",28782773-af58-4116-b5df-ee584f819ccb,fbsab-recruit-reef-fish-belt-transect-survey-a...,FBSAB RECRUIT Reef Fish Belt Transect Survey a...,Shore-based belt transects were conducted at 1...,active,dataset,None,...,5f4f1195-e770-4a2a-8f75-195cd98860ce,noaa-gov,National Oceanic and Atmospheric Administratio...,organization,,2020-11-10T15:36:13.098184,True,approved,active,"10367,belt transect survey,biology,central pac..."
3,bd3fb449-c524-4e6f-85a4-86fac4f1107b,https://catalog.data.gov/dataset/yellowknife-n...,"Archive of data.gov dataset ""Yellowknife, N. W...",5520ddd6-d0f6-463d-bd4d-4a734dbda691,yellowknife-n-w-t-nt-cyzf4,"Yellowknife, N. W. T., NT (CYZF)","Timeseries data from 'Yellowknife, N. W. T., N...",active,dataset,None,...,5f4f1195-e770-4a2a-8f75-195cd98860ce,noaa-gov,National Oceanic and Atmospheric Administratio...,organization,,2020-11-10T15:36:13.098184,True,approved,active,"aggregate_quality_flag,air_pressure_at_mean_se..."
4,ac27c8d9-33a2-4552-886a-c8b4e7418af1,https://catalog.data.gov/dataset/l02194-nos-hy...,"Archive of data.gov dataset ""L02194: NOS Hydro...",4410ddef-9a08-4b8c-ba12-03c190a54123,l02194-nos-hydrographic-survey,L02194: NOS Hydrographic Survey,The National Oceanic and Atmospheric Administr...,active,dataset,None,...,5f4f1195-e770-4a2a-8f75-195cd98860ce,noaa-gov,National Oceanic and Atmospheric Administratio...,organization,,2020-11-10T15:36:13.098184,True,approved,active,"bathymetry,bathymetry/seafloor topography,cont..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311815,38ac7f7f-f3a5-4680-940e-314919c38ac2,https://catalog.data.gov/dataset/ow-noaa-avhrr...,"Archive of data.gov dataset ""OW NOAA AVHRR-GAC...",a3e6e94a-c3f9-467a-8e42-82e34428c0b4,ow-noaa-avhrr-gac-sea-surface-temperature1,OW NOAA AVHRR-GAC Sea-Surface Temperature,The dataset contains satellite-derived sea-sur...,active,dataset,None,...,5f4f1195-e770-4a2a-8f75-195cd98860ce,noaa-gov,National Oceanic and Atmospheric Administratio...,organization,,2020-11-10T15:36:13.098184,True,approved,active,"3-day,avhrr,daily,doc/noaa/nmfs/pifsc,gac,glob..."
311816,4d52f844-35e4-4391-bf50-0920db67f3c9,https://catalog.data.gov/dataset/mora-county-b...,"Archive of data.gov dataset ""Mora County Block...",74567044-a1d7-421c-81e2-76bcd7576058,mora-county-blocks-age-by-5-year-age-groups-fo...,"Mora County Blocks, Age by 5-Year Age Groups f...",The once-a-decade decennial census was conduct...,active,dataset,None,...,6a3f

## Positives

In [6]:
drp = pd.read_parquet('00_data/03_03_drp_in_dgf_formodeling.parquet')


# Near deduplication

In [9]:
US_STATES = {
    'alabama','alaska','arizona','arkansas','california','colorado','connecticut',
    'delaware','florida','georgia','hawaii','idaho','illinois','indiana','iowa',
    'kansas','kentucky','louisiana','maine','maryland','massachusetts','michigan',
    'minnesota','mississippi','missouri','montana','nebraska','nevada',
    'new hampshire','new jersey','new mexico','new york','north carolina',
    'north dakota','ohio','oklahoma','oregon','pennsylvania','rhode island',
    'south carolina','south dakota','tennessee','texas','utah','vermont',
    'virginia','washington','west virginia','wisconsin','wyoming',
    'al','ak','az','ar','ca','co','ct','de','fl','ga','hi','id','il','in','ia',
    'ks','ky','la','me','md','ma','mi','mn','ms','mo','mt','ne','nv','nh','nj',
    'nm','ny','nc','nd','oh','ok','or','pa','ri','sc','sd','tn','tx','ut','vt',
    'va','wa','wv','wi','wy','dc',
}
MONTHS = {
    'january','february','march','april','may','june','july','august',
    'september','october','november','december',
    'jan','feb','mar','apr','jun','jul','aug','sep','oct','nov','dec',
}

In [10]:
def normalize_title(t):
    if not t or not isinstance(t, str): return ''
    t = t.lower().strip()
    t = re.sub(r'[-/\\|_]+', ' ', t)
    t = re.sub(r'[^a-z0-9\s]', '', t)
    return re.sub(r'\s+', ' ', t).strip()

# normalize descriptions
def normalize_desc(t):
    if not t or not isinstance(t, str):
        return ''
    t = t.lower()
    t = re.sub(r'<[^>]+>', ' ', t)
    t = re.sub(r'\b(19|20)\d{2}\b', '', t)
    t = re.sub(r'\bfy\s?\d{2,4}\b', '', t)
    t = re.sub(r'\bq[1-4]\b', '', t)
    t = re.sub(r'\bv\d+(\.\d+)*\b', '', t)
    t = re.sub(r'\b(version|edition|release|update|rev)\s*\d*\b', '', t)
    tokens = [tok for tok in t.split() if tok not in US_STATES and tok not in MONTHS]
    t = re.sub(r'[^a-z0-9\s]', '', ' '.join(tokens))
    return re.sub(r'\s+', ' ', t).strip()

In [11]:
# deduplicate per group
def dedup_group(group, threshold=0.7):
    if len(group) == 1:
        return group

    # pass 1: exact desc_norm match
    group = (
        group
        .sort_values('desc_len')
        .drop_duplicates('desc_norm', keep='first')
        .reset_index(drop=True)  # clean 0-based index for iloc below
    )
    if len(group) == 1:
        return group

    # pass 2: TF-IDF cosine on normalized descriptions
    descs = group['desc_norm'].tolist()
    if all(d.strip() == '' for d in descs):
        return group.iloc[[0]]

    tfidf = TfidfVectorizer(min_df=2, stop_words='english')
    try:
        X = tfidf.fit_transform(descs)
    except ValueError:
        return group.iloc[[0]]

    sims = cosine_similarity(X)

    kept = []
    absorbed = set()
    for i in range(len(group)):
        if i in absorbed:
            continue
        kept.append(i)
        for j in range(i + 1, len(group)):
            if sims[i, j] >= threshold:
                absorbed.add(j)

    cluster_map = {i: [i] for i in kept}
    for j in absorbed:
        for i in kept:
            if sims[i, j] >= threshold:
                cluster_map[i].append(j)
                break

    result_idxs = []
    for i, members in cluster_map.items():
        sub = group.iloc[members]
        med = sub['desc_len'].median()
        best = (sub['desc_len'] - med).abs().idxmin()
        result_idxs.append(best)

    return group.loc[result_idxs]


In [12]:
# normalize titles
def topicalize_title(t):
    if not t or not isinstance(t, str):
        return ''
    t = t.lower().strip()
    t = re.sub(r'\b(19|20)\d{2}\b', '', t)
    t = re.sub(r'\bfy\s?\d{2,4}\b', '', t)
    t = re.sub(r'\bq[1-4]\b', '', t)
    t = re.sub(r'\bv\d+(\.\d+)*\b', '', t)
    t = re.sub(r'\b(version|edition|release|update|rev)\s*\d*\b', '', t)
    tokens = [tok for tok in t.split() if tok not in US_STATES and tok not in MONTHS]
    t = re.sub(r'[^a-z0-9\s]', '', ' '.join(tokens))
    return re.sub(r'\s+', ' ', t).strip()

# dedup per topic
def topical_dedup(df, threshold=0.7):
    df = df.copy()
    df['desc_len']  = df['description'].fillna('').str.len()
    df['desc_norm'] = df['description'].fillna('').apply(normalize_desc)

    has_topic = df['title_topic'].str.strip() != ''
    with_topic = df[has_topic]
    without_topic = df[~has_topic]

    deduped = (
        with_topic
        .groupby('title_topic', group_keys=False)
        .apply(lambda g: dedup_group(g, threshold=threshold))
        .reset_index(drop=True)
    )

    result = pd.concat([deduped, without_topic], ignore_index=True)
    print(f'{len(df):,} -> {len(result):,}  (removed {len(df)-len(result):,})')
    return result


In [13]:
if 'title_norm' not in dgf.columns:
    dgf['title_norm'] = dgf['title'].fillna('').apply(normalize_title)

dgf['title_topic'] = dgf['title_norm'].apply(topicalize_title)

In [14]:
start = time.time()
dgf_dedup = topical_dedup(dgf, threshold=0.7)
print(f'Done in {time.time()-start:.1f}s')

514,352 -> 269,083  (removed 245,269)
Done in 1068.8s


In [15]:
dgf_dedup.to_parquet(str(DATA_DIR / '03_dgf_dedup.parquet'), index=False)
print(f'Sav                                                                                                                                                                                                                                                                                                 aed {len(dgf_dedup):,} rows  |  {dgf_dedup.shape[1]} cols')

Sav                                                                                                                                                                                                                                                                                                 aed 269,083 rows  |  32 cols


In [16]:
dgf_dedup = pd.read_parquet(str(DATA_DIR / '03_dgf_dedup.parquet'))
dgf_dedup

,title,description,publisher,identifier,slug,has_spatial,popularity,last_harvested_date,keyword,theme,...,dcat_spatial,dcat_bureau_code,dcat_program_code,dcat_periodicity,dcat_license,dcat_rights,title_norm,desc_len,desc_norm,title_topic
0,2013-005_299BPHOTOGRAPHS: SEABOSS Images from ...,"The U.S. Geological Survey (USGS), in cooperat...",U.S. Geological Survey,http://datainventory.doi.gov/id/dataset/USGS_1...,2013-005_299bphotographs-seaboss-images-from-t...,True,0,2026-01-27T03:12:54.564398,"[Atlantic Ocean, Block Island Sound, CMGP, Coa...",[Geospatial],...,"-72.096709, 41.125639, -71.858932, 41.193251",[010:12],[],,,,2013 005 299bphotographs seaboss images from t...,853,the us geological survey usgs cooperation with...,NaN
1,"006 - Santa Cruz Harbor, CA","Timeseries data from '006 - Santa Cruz Harbor,...",Axiom Data Science,edu_ucsd_cdip_006,006-santa-cruz-harbor-ca,True,2,2026-01-26T23:14:29.853735,"[earth science, atmosphere, ocean, biosphere, ...",[],...,"-122.0043,36.959633,-122.0043,36.959633",[],[],,https://creativecommons.org/publicdomain/zero/...,,006 santa cruz harbor ca,70,timeseries data from 006 santa cruz harbor ca ...,NaN
2,01) Revised groundwater-level contours for Smi...,These contours represent static groundwater le...,U.S. Geological Survey,http://datainventory.doi.gov/id/dataset/USGS_6...,01-revised-groundwater-level-contours-for-smit...,True,1,2026-01-27T03:18:03.546706,"[Great Basin, Groundwater, Lyon County, Mason ...",[Geospatial],...,"-119.4276, 38.7038, -119.0016, 39.1669",[010:12],[],,,,01 revised groundwater level contours for smit...,1352,these contours represent static groundwater le...,NaN
3,01. Seismicity catalogs for the 2023 update of...,This dataset contains earthquake catalogs comp...,U.S. Geological Survey,http://datainventory.doi.gov/id/dataset/USGS_6...,01-seismicity-catalogs-for-the-2023-update-of-...,True,3,2026-01-27T03:57:37.770642,"[Alaska, NSHM, NSHMP, National Seismic Hazard ...",[Geospatial],...,"170.0000, 49.5000, -128.0000, 72.0000",[010:12],[],,,,01 seismicity catalogs for the 2023 update of ...,536,this dataset contains earthquake catalogs comp...,NaN
4,01: Watersheds shapefile for the 15 study wate...,This Geographic Information System dataset con...,U.S. Geological Survey,http://datainventory.doi.gov/id/dataset/USGS_6...,01-watersheds-shapefile-for-the-15-study-water...,True,0,2026-01-27T02:17:38.778671,"[Gwinnett County, State of Georgia, USGS:63502...",[Geospatial],...,"-84.2721, 33.7676, -83.8468, 34.1514",[010:12],[],,,,01 watersheds shapefile for the 15 study water...,234,this geographic information system dataset con...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269078,Wyoming (WY),This report presents Wyoming\'s estimates for ...,Substance Abuse and Mental Health Services Adm...,https://healthdata.gov/api/views/7756-egub,wyoming-wy-1b2b8,False,0,2025-09-07T09:21:31.075371,"[mental-health, samhsa, substance-use, suicida...",[SAMHSA],...,,[009:30],[009:061],,,,wyoming wy,600,this report presents wyomings estimates for 25...,
269079,Missouri (MO),This report uses 2008 to 2010 National Survey ...,Substance Abuse and Mental Health Services Adm...,https://healthdata.gov/api/views/36za-n73h,missouri-mo,False,0,2025-09-07T07:24:52.464410,"[mental-health-indicators, missouri, nsduh-dat...",[SAMHSA],...,,[009:30],[009:061],,,,missouri mo,536,this report uses to national survey on drug us...,
269080,Maine (ME),This report presents Maine\'s estimates for 25...,Substance Abuse and Mental Health Services Adm...,https://healthdata.gov/api/views/cmqr-hkdu,maine-me-b7fd4,False,0,2025-09-07T12:17:17.923100,"[maine, mental-health, nsduh, samhsa, substanc...",[SAMHSA],...,,[009:30],[009:061],,,,maine me,596,this report presents maines estimates for 25 m...,
269081,Nebraska (NE),This report presents estimates for measures of...,Substance Abuse and Mental Health Services Adm...,https://healthdata.gov/api/views/nbxn-azk7,nebraska-ne-d1321

In [17]:
dgf_dedup['desc_len'] = dgf_dedup['description'].fillna('').str.len() 

In [18]:
dgf['desc_len'] = dgf['description'].fillna('').str.len() 

## Checks

what got collapsed?

In [23]:
removed_titles = set(dgf['title_norm']) - set(dgf_dedup['title_norm'])

In [24]:
sample = (dgf[dgf['title_norm'].isin(removed_titles)].groupby('title_topic', group_keys=False).apply(lambda g: g.sample(min(2, len(g)), random_state=0)).head(20))
display(sample[['title', 'org_name', 'desc_len']].reset_index(drop=True))

,title,org_name,desc_len
0,1-meter Digital elevation model (DEM) of beach...,Department of the Interior,607
1,10-meter Digital elevation model (DEM) of beac...,Department of the Interior,821
2,1.01 ALS Response Time (2023),City of Tempe,1752
3,1.01 ALS Response Time (2014),City of Tempe,2034
4,1970's Land use data refined with 2000 populat...,Department of the Interior,400
5,2-foot Elevation contours of beach topography ...,Department of the Interior,807
6,2MASS ASTEROID AND COMET SURVEY V2.0,National Aeronautics and Space Administration,245
7,2MASS ASTEROID AND COMET SURVEY V2.0,National Aeronautics and Space Administration,245
8,2002 3 ft Digital Elevation Model of Clarksbur...,Department of the Interior,246
9,2018 3 ft Digital Elevation Model of Clarksbur...,Department of the Interior,246


how sensitive is the result to threshold?

In [25]:
for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    out = topical_dedup(dgf, threshold=t)
    removed_pct = (len(dgf) - len(out)) / len(dgf) * 100
    print(f'threshold={t:.1f}  ->  {len(out):,} kept  ({removed_pct:.1f}% removed)')

514,352 -> 268,660  (removed 245,692)
threshold=0.5  ->  268,660 kept  (47.8% removed)


KeyboardInterrupt: 

removed vs kept

In [26]:
removed_df = dgf[~dgf['title_norm'].isin(dgf_dedup['title_norm'])].copy()
keeper_lookup = (dgf_dedup.drop_duplicates('title_topic').set_index('title_topic')[['title', 'description', 'org_name']].rename(columns=lambda c: f'kept_{c}'))

In [27]:
pairs_df = (
    removed_df[['title', 'description', 'org_name', 'title_topic']]
    .rename(columns=lambda c: f'removed_{c}' if c != 'title_topic' else c)
    .join(keeper_lookup, on='title_topic')
    .dropna(subset=['kept_title'])
)

In [28]:
pairs_df['cross_org'] = pairs_df['kept_org_name'] != pairs_df['removed_org_name']
pairs_df['kept_desc'] = pairs_df['kept_description'].str[:120]
pairs_df['removed_desc'] = pairs_df['removed_description'].str[:120]

print(f'{len(pairs_df):,} removed records  |  cross-org: {pairs_df["cross_org"].sum()}')

0 removed records  |  cross-org: 0


In [29]:
display_cols = ['cross_org', 'kept_title', 'removed_title', 'kept_desc', 'removed_desc', 'kept_org_name', 'removed_org_name']
display(
    pd.concat([
        pairs_df[pairs_df['cross_org']],
        pairs_df[~pairs_df['cross_org']].sample(min(15, (~pairs_df['cross_org']).sum()), random_state=0),
    ])[display_cols].reset_index(drop=True)
)

,cross_org,kept_title,removed_title,kept_desc,removed_desc,kept_org_name,removed_org_name


to confirm the other side: same title groups where records were correctly kept distinct

In [30]:
topic_counts_before = dgf.groupby('title_topic').size()
topic_counts_after  = dgf_dedup.groupby('title_topic').size()

still_multiple = topic_counts_after[topic_counts_after > 1].index

topics with >1 record after dedup - different descriptions

In [31]:
for topic in still_multiple[:5]:
    group = dgf_dedup[dgf_dedup['title_topic'] == topic]
    print(f'"\n{topic}"  ({len(group)} records kept)')
    for _, row in group.iterrows():
        print(f'  {row["title"]}')
        print(f'  {str(row["description"])[:150]}')

"
"  (116 records kept)
  Oregon (OR)
  This report presents Oregon\'s estimates for 25 measures of substance use and mental health based on the combined 2009 and 2010 National Surveys on Dr
  DC 2050
  DC 2050 presents an opportunity for the District to identify future challenges and opportunities and consider how to meet them in the next two decades
  Texas (TX)
  This report presents Texas\'s estimates for 25 measures of substance use and mental health based on the combined 2009 and 2010 National Surveys on Dru
  Utah (UT)
  This report uses 2008 to 2010 National Survey on Drug Use and Health (NSDUH) to assess past year alcohol use disorder and illicit drug use disorder am
  Idaho (ID)
  This report uses 2008 to 2010 National Survey on Drug Use and Health (NSDUH) to assess past year alcohol use disorder and illicit drug use disorder am
  Nevada (NV)
  This report presents Nevada\'s estimates for 25 measures of substance use and mental health based on the combined 2009 and 2010 Natio

org-type breakdown after dedup

In [33]:
print(dgf_dedup['org_type'].value_counts().head(15).to_string())

org_type
Federal Government    248450
State Government        9612
City Government         8395
County Government       2495
University                80
                          51


# Next: 04_embeddings